# SIMSOPT Stage-2 Coils

## What you will learn
Why a good plasma boundary is incomplete without coil diagnostics.

## Codes used
SIMSOPT in research mode; synthetic coil and `B dot n` fallback by default.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`04_initial_coils.png`, `04_final_coils.png`, `04_bdotn_before_after.png`, and `04_coil_tradeoff.png`.


In [ ]:

from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Use cached mode first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")


In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:

import matplotlib.pyplot as plt
from sos2026.coil_helpers import coil_curves, normal_field_error, tradeoff_table
from sos2026.plotting import savefig, caption
for stage, name in [("initial", "04_initial_coils.png"), ("final", "04_final_coils.png")]:
    fig = plt.figure(figsize=(5.8, 4.2))
    ax = fig.add_subplot(111, projection="3d")
    for curve in coil_curves(stage):
        ax.plot(curve[:,0], curve[:,1], curve[:,2])
    ax.set_title(f"{stage} coils")
    ax.set_axis_off()
    savefig(fig, name)
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6))
for ax, stage in zip(axes, ["initial", "final"]):
    theta, zeta, err = normal_field_error(stage)
    im = ax.contourf(zeta, theta, err, levels=20, cmap="coolwarm")
    ax.set_title(stage)
    ax.set_xlabel("zeta")
    ax.set_ylabel("theta")
fig.colorbar(im, ax=axes.ravel().tolist(), label="B dot n proxy")
savefig(fig, "04_bdotn_before_after.png")
table = tradeoff_table()
fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(table["coil_length"], table["quadratic_flux"], marker="o")
ax.set_xlabel("coil length")
ax.set_ylabel("quadratic flux proxy")
ax.set_title("Coil tradeoff")
ax.grid(alpha=0.25)
caption(ax, "Shorter coils are easier, but can damage the target field.")
savefig(fig, "04_coil_tradeoff.png")
print("Caption: stage-2 optimization is a tradeoff between field accuracy and manufacturable coils.")


## Try this
Increase the length penalty in the table and decide which metric gets worse.

## Expected qualitative answer
The coils become easier, but the normal-field error generally rises.

## Research extension
Load `input.LandremanPaul2021_QA` with SIMSOPT and replay a tiny stage-2 optimization.
